# 10 Power and sample size: Cohen's tables, G*Power, Stata, presize, pwr and R

`sdata.find_how_many_needed` and `sdata.find_chance_of_finding` on worked examples that were printed in books,
manuals and papers, and against the packages that compute the same thing:

- **published values**: Cohen (1988), *Statistical Power Analysis for the Behavioral Sciences*, 2nd ed., Table
  2.4.1 (values per group for the two-sample t-test) and Table 8.4.4 (one-way ANOVA); Faul et al. (2007) and the
  G*Power 3.1 manual; the help pages of R's `power.t.test` and `power.prop.test`; the vignette of R pwr; the
  Stata power manuals ([PSS-2] power logrank, power twomeans cluster); the presize paper (Haynes et al. 2021);
  Bland's note on cluster trials. These are worked examples (facts); no data are stored.
- **the Python parent**: statsmodels' power classes (`TTestIndPower`, `TTestPower`, `FTestAnovaPower`,
  `GofChisquarePower`, `FTestPowerF2`), where simplydata calls them; SciPy `stats.power` with the same seed for the
  rank test, which simplydata simulates with it.
- **R**: `power.t.test` and `power.prop.test` (both run with `tol = 1e-12`, much tighter than their default), pwr
  (`pwr.anova.test`, `pwr.chisq.test`, `pwr.r.test`, `pwr.f2.test`), TOSTER `power_t_TOST`, presize, gsDesign
  `nEvents`, powerSurvEpi and pwrss.

**Only simplydata runs in this notebook.** The reference values (`references/10_power_and_sample_size.csv`, from
`scripts/references/10_power_and_sample_size.py`): the printed values first, then the packages run once. Every
check prints its source.

Tolerances (the same for every example in this notebook):

- a whole number of values (rounded up, as every program prints it): exact;
- a power at a given number of values, or a number of events from a closed formula: relative difference 1e-10;
- an exact (unrounded) number of values, which every program finds by a root search: relative difference 1e-6.
  pwr finds n with R's `uniroot` at its default tolerance (about 1e-4 in n), so for pwr the exact n is solved
  by the reference script from pwr's own power function with `uniroot(tol = 1e-12)`;
- a printed value: half a unit of its last printed digit.

A power "with that many" is the reference's power at its own whole number of values (the ceiling of its exact n),
which simplydata's whole number is checked against.

In [1]:
import math

import simplydata as sdata

from checks import ReferenceChecks, quiet, read_row, versions

NOTEBOOK = "10_power_and_sample_size"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)


def row(word, **options):
    """The one-row answer of a simplydata power word, without its printout (table.get_row)."""
    return read_row(quiet(word, **options))


def check_row(dataset, method, found, columns):
    """Check the columns of a simplydata answer that have a reference; columns maps quantity -> column."""
    checks.check_values(dataset, method, {quantity: found[column] for quantity, column in columns.items()})

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit 3f0192b of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## Two averages: the two-sample t-test

The full printed answer for Cohen's medium difference (d = 0.5, 80 % power, two-sided 5 %):

In [2]:
sdata.find_how_many_needed(effect_size=0.5)

To find a difference of 0.5 standard deviations (Cohen's d) with an 80 % chance, using a two-sided test at 0.05, you need 64 values in each group (128 in all). Method: two-sample t-test power, as R power.t.test().
Note: the calculation uses the usual equal-spread formula; table.compare_averages_of uses Welch's t-test, which is very close to it when the groups are the same size.
Cohen's rough guide calls this a medium effect (Cohen's d: 0.2 small, 0.5 medium, 0.8 large); it is only a hint, sizes from your own field are better.


,test,method,R function,effect size name,effect size,groups,sides,significance,power wanted,values per group (exact),values per group,values in all,power with that many
1,compare_averages,two-sample t-test power,power.t.test(),Cohen's d,0.5,2,2,0.05,0.8,63.765764,64,128,0.801459


Cohen (1988, Table 2.4.1) prints the values per group for 80 % power at a two-sided 5 % level: 393 for
d = 0.2, 64 for d = 0.5 and 26 for d = 0.8. Faul et al. (2007, G*Power 3, Behavior Research Methods 39, 175-191)
give 105 per group (210 in all) for d = 0.5 at 95 % power. The help page of R `power.t.test` prints
`power.t.test(n = 20, delta = 1)`: power = 0.8689528, and `power.t.test(power = .90, delta = 1)`: n = 22.02110.
simplydata's two-sided power counts only the right direction, as R's default (`strict = FALSE`); statsmodels
gives that as the one-sided power at half the level.

In [3]:
TWO = {"values per group (exact)": "values per group (exact)", "values per group": "values per group",
       "power with that many": "power with that many"}


def t_two(d, power=0.8, sides=2, label=None, difference=None, spread=None):
    options = dict(power=power, sides=sides)
    options.update(dict(difference=difference, spread=spread) if difference is not None else dict(effect_size=d))
    found = row(sdata.find_how_many_needed, **options)
    check_row(label or f"d = {d}, power {power}, {sides}-sided", "find_how_many_needed (two averages)", found, TWO)


for d in (0.2, 0.5, 0.8):
    t_two(d)
t_two(0.5, power=0.95)
t_two(1.0, power=0.9)
t_two(0.5, sides=1)
t_two(0.5, power=0.9, label="difference 5, spread 10, power 0.9", difference=5, spread=10)

# the power for a given number of values
for d, n in ((1.0, 20), (0.5, 30), (0.3, 100)):
    found = row(sdata.find_chance_of_finding, effect_size=d, values_per_group=n)
    checks.check(f"d = {d}, {n} per group", "find_chance_of_finding (two averages)", "power", found["power"])

agrees  d = 0.2, power 0.8, 2-sided | find_how_many_needed (two averages) | values per group (exact): simplydata 393.406654178, statsmodels TTestIndPower 393.406654178; relative difference 1.44e-16 (allowed 1e-06)
      Source: statsmodels TTestIndPower (run reference) - https://www.statsmodels.org/v0.15.0/search.html?q=TTestIndPower
agrees  d = 0.2, power 0.8, 2-sided | find_how_many_needed (two averages) | values per group (exact): simplydata 393.406654178, R power.t.test 393.406656228; relative difference 5.21e-09 (allowed 1e-06)
      Source: R power.t.test (run reference) - https://search.r-project.org/R/refmans/stats/html/power.t.test.html
agrees  d = 0.2, power 0.8, 2-sided | find_how_many_needed (two averages) | values per group: simplydata 394, R power.t.test 394; difference 0.00e+00 (allowed 0e+00)
      Source: R power.t.test (run reference) - https://search.r-project.org/R/refmans/stats/html/power.t.test.html
explained difference – reference (C7)  d = 0.2, power 0.8, 2-side

## One group, and before and after: the one-sample and paired t-tests

The same formula with one group of values or of differences (statsmodels `TTestPower`; R
`power.t.test(type = "one.sample")` and `type = "paired"`).

In [4]:
for test in ("compare_average_to", "compare_before_after"):
    for d in (0.3, 0.5):
        found = row(sdata.find_how_many_needed, test=test, effect_size=d)
        check_row(f"d = {d}, power 0.8", f"find_how_many_needed ({test})", found,
                  {"values (exact)": "values per group (exact)", "values": "values per group",
                   "power with that many": "power with that many"})

agrees  d = 0.3, power 0.8 | find_how_many_needed (compare_average_to) | values (exact): simplydata 89.149577633, statsmodels TTestPower 89.149577633; relative difference 1.59e-16 (allowed 1e-06)
      Source: statsmodels TTestPower (run reference) - https://www.statsmodels.org/v0.15.0/search.html?q=TTestPower
agrees  d = 0.3, power 0.8 | find_how_many_needed (compare_average_to) | values (exact): simplydata 89.149577633, R power.t.test (one.sample) 89.1495772662; relative difference 4.11e-09 (allowed 1e-06)
      Source: R power.t.test (one.sample) (run reference) - https://search.r-project.org/R/refmans/stats/html/power.t.test.html
agrees  d = 0.3, power 0.8 | find_how_many_needed (compare_average_to) | values: simplydata 90, R power.t.test (one.sample) 90; difference 0.00e+00 (allowed 0e+00)
      Source: R power.t.test (one.sample) (run reference) - https://search.r-project.org/R/refmans/stats/html/power.t.test.html
agrees  d = 0.3, power 0.8 | find_how_many_needed (compare_average

## Three or more averages: one-way ANOVA with Cohen's f

Cohen (1988, Table 8.4.4) prints n = 45 per group for f = 0.25 with 4 groups (u = 3) at 80 % power and a 5 %
level. statsmodels `FTestAnovaPower` takes all the values (groups x n); R `pwr.anova.test` takes n per group.

In [5]:
for f, k in ((0.25, 4), (0.4, 3), (0.1, 5)):
    found = row(sdata.find_how_many_needed, effect_size=f, groups=k)
    check_row(f"f = {f}, {k} groups, power 0.8", "find_how_many_needed (ANOVA)", found, TWO)

agrees  f = 0.25, 4 groups, power 0.8 | find_how_many_needed (ANOVA) | values per group (exact): simplydata 44.5992743068, statsmodels FTestAnovaPower 44.5992743068; relative difference 0.00e+00 (allowed 1e-06)
      Source: statsmodels FTestAnovaPower (run reference) - https://www.statsmodels.org/v0.15.0/search.html?q=FTestAnovaPower
agrees  f = 0.25, 4 groups, power 0.8 | find_how_many_needed (ANOVA) | values per group (exact): simplydata 44.5992743068, R pwr.anova.test power, solved by uniroot (tol 1e-12) 44.5992742847; relative difference 4.94e-10 (allowed 1e-06)
      Source: R pwr.anova.test power, solved by uniroot (tol 1e-12) (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/10_power_and_sample_size.py
agrees  f = 0.25, 4 groups, power 0.8 | find_how_many_needed (ANOVA) | values per group: simplydata 45, R pwr.anova.test 45; difference 0.00e+00 (allowed 0e+00)
      Source: R pwr.anova.test (run reference) - https://github.com/biof

## Counts: the chi-square test with Cohen's w, and two percents

`test="compare_counts"` with `groups=(groups, kinds of outcome)`: R `pwr.chisq.test` (N = all values) and
statsmodels `GofChisquarePower` (with df + 1 bins). With `percents=[p1, p2]`: R `power.prop.test`, whose help page
prints `power.prop.test(p1 = .50, p2 = .75, power = .90)`: n = 76.70693 per group, and
`power.prop.test(n = 50, p1 = .50, p2 = .75)`: power = 0.7401659.

In [6]:
for w, groups in ((0.3, (2, 2)), (0.3, (3, 4)), (0.1, (2, 3))):
    found = row(sdata.find_how_many_needed, test="compare_counts", effect_size=w, groups=groups)
    check_row(f"w = {w}, {groups[0]} x {groups[1]} table, power 0.8", "find_how_many_needed (chi-square)", found, TWO)

for p1, p2, power in ((50, 75, 0.9), (30, 15, 0.8), (10, 5, 0.8)):
    found = row(sdata.find_how_many_needed, test="compare_counts", percents=[p1, p2], power=power)
    check_row(f"{p1} % against {p2} %, power {power}", "find_how_many_needed (two percents)", found, TWO)
found = row(sdata.find_chance_of_finding, test="compare_counts", percents=[50, 75], values_per_group=50)
checks.check("50 % against 75 %, 50 per group", "find_chance_of_finding (two percents)", "power", found["power"])

agrees  w = 0.3, 2 x 2 table, power 0.8 | find_how_many_needed (chi-square) | values per group (exact): simplydata 43.6047806907, statsmodels GofChisquarePower 43.6047806907; relative difference 0.00e+00 (allowed 1e-06)
      Source: statsmodels GofChisquarePower (run reference) - https://www.statsmodels.org/v0.15.0/search.html?q=GofChisquarePower
agrees  w = 0.3, 2 x 2 table, power 0.8 | find_how_many_needed (chi-square) | values per group (exact): simplydata 43.6047806907, R pwr.chisq.test power, solved by uniroot (tol 1e-12) 43.6047806074; relative difference 1.91e-09 (allowed 1e-06)
      Source: R pwr.chisq.test power, solved by uniroot (tol 1e-12) (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/10_power_and_sample_size.py
agrees  w = 0.3, 2 x 2 table, power 0.8 | find_how_many_needed (chi-square) | power with that many: simplydata 0.803527484456, statsmodels GofChisquarePower 0.803527484456; relative difference 0.00e+00 (allowed 1e

2

## A correlation: R pwr.r.test (Fisher's z)

statsmodels has no power class for a correlation, so R pwr is the only reference.

In [7]:
for r_size in (0.1, 0.3, 0.5):
    found = row(sdata.find_how_many_needed, test="correlation_between", effect_size=r_size)
    check_row(f"r = {r_size}, power 0.8", "find_how_many_needed (correlation)", found,
              {"pairs (exact)": "values per group (exact)", "pairs": "values per group",
               "power with that many": "power with that many"})
    found = row(sdata.find_chance_of_finding, test="correlation_between", effect_size=r_size, values_per_group=50)
    checks.check(f"r = {r_size}, 50 pairs", "find_chance_of_finding (correlation)", "power", found["power"])

agrees  r = 0.1, power 0.8 | find_how_many_needed (correlation) | pairs (exact): simplydata 781.751562817, R pwr.r.test power, solved by uniroot (tol 1e-12) 781.751564249; relative difference 1.83e-09 (allowed 1e-06)
      Source: R pwr.r.test power, solved by uniroot (tol 1e-12) (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/10_power_and_sample_size.py
agrees  r = 0.1, power 0.8 | find_how_many_needed (correlation) | pairs: simplydata 782, R pwr.r.test 782; difference 0.00e+00 (allowed 0e+00)
      Source: R pwr.r.test (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/10_power_and_sample_size.py
agrees  r = 0.1, power 0.8 | find_how_many_needed (correlation) | power with that many: simplydata 0.800124937742, R pwr.r.test 0.800124937742; relative difference 0.00e+00 (allowed 1e-10)
      Source: R pwr.r.test (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scrip

## Regression: Cohen's f² (statsmodels FTestPowerF2, R pwr.f2.test)

The G*Power 3.1 manual (section 13.3.1, Cohen 1988 example 9.1): 5 columns, R squared 0.10 (f² = 0.1 / 0.9),
95 rows: power 0.673586. The pwr vignette (`pwr.f2.test(u = 2, f2 = 0.3 / 0.7, sig.level = 0.001, power = 0.8)`,
Kutner et al. exercise 8.43): v = 49.88971, so 49.88971 + 3 + 1 rows for 2 columns added to 1.

In [8]:
found = row(sdata.find_chance_of_finding, test="explain", effect_size=0.1 / 0.9, using=5, values_per_group=95)
checks.check("f² = 0.1/0.9, 5 columns, 95 rows", "find_chance_of_finding (regression)", "power", found["power"])

found = row(sdata.find_how_many_needed, test="explain", effect_size=0.15, using=["a", "b", "c"])
checks.check("f² = 0.15, 3 columns, power 0.8", "find_how_many_needed (regression)", "rows (exact)", found["rows (exact)"])

found = row(sdata.find_how_many_needed, test="compare_models", effect_size=0.3 / 0.7, using=["x1"],
            against=["x1", "x2", "x3"], significance=0.001)
checks.check("2 columns added to 1, f² = 0.3/0.7, significance 0.001", "find_how_many_needed (compare_models)",
             "rows (exact)", found["rows (exact)"])

agrees  f² = 0.1/0.9, 5 columns, 95 rows | find_chance_of_finding (regression) | power: simplydata 0.673585770819, G*Power 3.1 manual 13.3.1, printed 0.673586; difference 2.29e-07 (allowed 5e-07)
      Source: G*Power 3.1 manual 13.3.1, printed (printed) - https://www.psychologie.hhu.de/fileadmin/redaktion/Fakultaeten/Mathematisch-Naturwissenschaftliche_Fakultaet/Psychologie/AAP/gpower/GPowerManual.pdf
agrees  f² = 0.1/0.9, 5 columns, 95 rows | find_chance_of_finding (regression) | power: simplydata 0.673585770819, statsmodels FTestPowerF2 0.673585770819; relative difference 1.65e-16 (allowed 1e-10)
      Source: statsmodels FTestPowerF2 (run reference) - https://www.statsmodels.org/v0.15.0/search.html?q=FTestPowerF
explained difference – reference (C7)  f² = 0.1/0.9, 5 columns, 95 rows | find_chance_of_finding (regression) | power: simplydata 0.673585770819, R pwr.f2.test 0.673585770914; relative difference 1.42e-10 (allowed 1e-10)
      why: notebook 10: Cohen (1988) Table 2.4.1 at d

2

## Logistic regression, equivalence and the width of an interval

- Logistic regression with one normal column, odds ratio 1.5 per standard deviation, 50 % at the average, 95 %
  power: the G*Power 3.1 manual (section 29.4) prints 337 (Demidenko 2007 with the variance correction); R
  `pwrss::power.z.logistic(method = "demidenko(vc)")`.
- Equivalence of two averages within plus or minus 5 (spread 10, no true difference): R `TOSTER::power_t_TOST`.
- A 95 % interval of an average 4 wide (spread 10): R `presize::prec_mean`; a Wilson interval of a percent near
  15 % that is 5 points wide: presize prints 783.4897, so 784 values (Haynes et al. 2021, JOSS 6, 3118).

In [9]:
found = row(sdata.find_how_many_needed, test="explain", outcome="yes or no", using=["x"], percents=[50],
            effect_size=1.5, power=0.95)
check_row("odds ratio 1.5, 50 %, power 0.95", "find_how_many_needed (logistic regression)", found,
          {"values": "values", "values (exact)": "values (exact)"})

found = row(sdata.find_how_many_needed, same_within=5, spread=10)
check_row("within 5, spread 10, power 0.8", "find_how_many_needed (equivalence, TOST)", found, TWO)

found = row(sdata.find_how_many_needed, interval_width=4, spread=10)
checks.check("interval 4 wide, spread 10", "find_how_many_needed (interval width)", "values (exact)",
             found["values (exact)"])
found = row(sdata.find_how_many_needed, interval_width=5, percents=[15])
check_row("Wilson interval 5 points wide near 15 %", "find_how_many_needed (interval width)", found,
          {"values (exact)": "values (exact)", "values": "values"})

agrees  odds ratio 1.5, 50 %, power 0.95 | find_how_many_needed (logistic regression) | values: simplydata 337, G*Power 3.1 manual 29.4, printed 337; difference 0.00e+00 (allowed 0e+00)
      Source: G*Power 3.1 manual 29.4, printed (printed) - https://www.psychologie.hhu.de/fileadmin/redaktion/Fakultaeten/Mathematisch-Naturwissenschaftliche_Fakultaet/Psychologie/AAP/gpower/GPowerManual.pdf
agrees  odds ratio 1.5, 50 %, power 0.95 | find_how_many_needed (logistic regression) | values (exact): simplydata 336.86477921, R pwrss::power.z.logistic 336.86477921; relative difference 0.00e+00 (allowed 1e-06)
      Source: R pwrss::power.z.logistic (run reference) - https://search.r-project.org/CRAN/refmans/pwrss/html/power.z.logistic.html (version 1.3.3)


agrees  within 5, spread 10, power 0.8 | find_how_many_needed (equivalence, TOST) | values per group (exact): simplydata 69.1978220019, R TOSTER::power_t_TOST 69.1978220019; relative difference 0.00e+00 (allowed 1e-06)
      Source: R TOSTER::power_t_TOST (run reference) - https://search.r-project.org/CRAN/refmans/TOSTER/html/power_t_TOST.html (version 0.8.6)
agrees  within 5, spread 10, power 0.8 | find_how_many_needed (equivalence, TOST) | values per group: simplydata 70, R TOSTER::power_t_TOST 70; difference 0.00e+00 (allowed 0e+00)
      Source: R TOSTER::power_t_TOST (run reference) - https://search.r-project.org/CRAN/refmans/TOSTER/html/power_t_TOST.html (version 0.8.6)
agrees  within 5, spread 10, power 0.8 | find_how_many_needed (equivalence, TOST) | power with that many: simplydata 0.805931181627, R TOSTER::power_t_TOST 0.805931181627; relative difference 0.00e+00 (allowed 1e-10)
      Source: R TOSTER::power_t_TOST (run reference) - https://search.r-project.org/CRAN/refmans/T

## Survival: events by Schoenfeld's formula, people by Freedman's

The Stata manual ([PSS-2] power logrank) prints 66 events for a hazard ratio of 0.5 (example 2), 266 events for
survival 50 % against 60 % one-sided (example 5, from Machin et al. 2009, p. 91), and 36 people per group by
Freedman's formula when everyone is followed to the event (example 1). R `gsDesign::nEvents` (Schoenfeld) and
`powerSurvEpi` (`ssizeEpi.default` for the people, `ssizeCT.default` for Freedman).

In [10]:
for hr, sides in ((0.5, 2), (math.log(0.5) / math.log(0.6), 1), (0.7, 2)):
    found = row(sdata.find_how_many_needed, test="compare_survival", effect_size=hr, sides=sides)
    check_row(f"hazard ratio {hr:.4g}, {sides}-sided", "find_how_many_needed (survival, events)", found,
              {"events": "events", "events (exact)": "events (exact)"})

found = row(sdata.find_how_many_needed, test="compare_survival", effect_size=0.7, percents=[30, 40])
checks.check("hazard ratio 0.7, 30 % and 40 % with the event", "find_how_many_needed (survival, people)",
             "people in all", found["people in all"])

found = row(sdata.find_how_many_needed, test="compare_survival", effect_size=0.5, percents=[100, 100],
            method="freedman")
checks.check("hazard ratio 0.5, everyone followed to the event", "find_how_many_needed (survival, Freedman)",
             "people per group", found["people per group"])

agrees  hazard ratio 0.5, 2-sided | find_how_many_needed (survival, events) | events: simplydata 66, Stata [PSS-2] power logrank, printed 66; difference 0.00e+00 (allowed 0e+00)
      Source: Stata [PSS-2] power logrank, printed (printed) - https://www.stata.com/manuals/psspowerlogrank.pdf
agrees  hazard ratio 0.5, 2-sided | find_how_many_needed (survival, events) | events: simplydata 66, R gsDesign::nEvents 66; difference 0.00e+00 (allowed 0e+00)
      Source: R gsDesign::nEvents (run reference) - https://search.r-project.org/CRAN/refmans/gsDesign/html/nEvents.html (version 3.11.0)
agrees  hazard ratio 0.5, 2-sided | find_how_many_needed (survival, events) | events (exact): simplydata 65.3456592589, R gsDesign::nEvents 65.3456592589; relative difference 0.00e+00 (allowed 1e-10)
      Source: R gsDesign::nEvents (run reference) - https://search.r-project.org/CRAN/refmans/gsDesign/html/nEvents.html (version 3.11.0)
agrees  hazard ratio 1.357, 1-sided | find_how_many_needed (survival, ev

agrees  hazard ratio 0.7, 30 % and 40 % with the event | find_how_many_needed (survival, people) | people in all: simplydata 706, R powerSurvEpi::ssizeEpi.default 706; difference 0.00e+00 (allowed 0e+00)
      Source: R powerSurvEpi::ssizeEpi.default (run reference) - https://search.r-project.org/CRAN/refmans/powerSurvEpi/html/ssizeEpi.default.html (version 0.1.5)
agrees  hazard ratio 0.5, everyone followed to the event | find_how_many_needed (survival, Freedman) | people per group: simplydata 36, Stata [PSS-2] power logrank, printed 36; difference 0.00e+00 (allowed 0e+00)
      Source: Stata [PSS-2] power logrank, printed (printed) - https://www.stata.com/manuals/psspowerlogrank.pdf
agrees  hazard ratio 0.5, everyone followed to the event | find_how_many_needed (survival, Freedman) | people per group: simplydata 36, R powerSurvEpi::ssizeCT.default 36; difference 0.00e+00 (allowed 0e+00)
      Source: R powerSurvEpi::ssizeCT.default (run reference) - https://search.r-project.org/CRAN/r

2

## Clusters, and a rank test by simulation

- Two averages in clusters of 20 with intracluster correlation 0.025, difference 1.1, spread 3.67: the Stata manual
  ([PSS-2] power twomeans, cluster, example 1, from Ahn et al. 2015) prints 13 clusters per group, 260 values. The
  design effect is 1 + (m - 1) ICC (Kerry and Bland 1998); Bland's note prints 2.45 for clusters of 30 with ICC
  0.05.
- `based_on="ranks"`: the Mann-Whitney test, power by simulation with SciPy `stats.power` (10,000 tries, seed 1).
  SciPy with the same generator and seed is the parent and must give the same power (1e-10).

In [11]:
found = row(sdata.find_how_many_needed, difference=1.1, spread=3.67, in_clusters_of=20, alike_within_clusters=0.025)
check_row("clusters of 20, ICC 0.025", "find_how_many_needed (clusters)", found,
          {"clusters per group": "clusters per group", "values in whole clusters": "values in whole clusters",
           "design effect": "design effect"})
found = row(sdata.find_how_many_needed, effect_size=0.5, in_clusters_of=30, alike_within_clusters=0.05)
checks.check("clusters of 30, ICC 0.05", "find_how_many_needed (clusters)", "design effect", found["design effect"])

found = row(sdata.find_chance_of_finding, effect_size=0.5, values_per_group=64, based_on="ranks")
checks.check("d = 0.5, 64 per group", "find_chance_of_finding (ranks, simulated)", "power", found["power"])

agrees  clusters of 20, ICC 0.025 | find_how_many_needed (clusters) | clusters per group: simplydata 13, Stata [PSS-2] power twomeans cluster, printed 13; difference 0.00e+00 (allowed 0e+00)
      Source: Stata [PSS-2] power twomeans cluster, printed (printed) - https://www.stata.com/manuals/psspowertwomeanscluster.pdf
agrees  clusters of 20, ICC 0.025 | find_how_many_needed (clusters) | values in whole clusters: simplydata 260, Stata [PSS-2] power twomeans cluster, printed 260; difference 0.00e+00 (allowed 0e+00)
      Source: Stata [PSS-2] power twomeans cluster, printed (printed) - https://www.stata.com/manuals/psspowertwomeanscluster.pdf
agrees  clusters of 20, ICC 0.025 | find_how_many_needed (clusters) | design effect: simplydata 1.475, 1 + (m - 1) ICC 1.475; relative difference 0.00e+00 (allowed 1e-10)
      Source: 1 + (m - 1) ICC (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/10_power_and_sample_size.py
agrees  clusters of 30, 

1

## Result

In [12]:
results = checks.finish()

10_power_and_sample_size: 151 comparisons - 146 agrees, 5 explained difference – reference
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)
